# 03 – Airports & flights
Finds the airports near each city and tomorrow's arrivals (AeroDataBox via RapidAPI), and loads them into the MySQL tables `airports` and `flights`.

- `airports`: one row per airport, `airport_icao` (official 4-letter code) as primary key, linked to `cities` by `city_id`
- `flights`: one row per landing plane (codeshares, cargo and private flights excluded), linked to `airports` by `arrival_airport_icao`

## 1. Setup

In [2]:
# --- libraries ---
import os                          # access settings of the operating system (used to read .env values)
import requests                    # send requests to APIs over the internet
import pandas as pd                # tables (DataFrames) + read_sql / to_sql for MySQL
from dotenv import load_dotenv     # reads the .env file that holds our secrets

# --- secrets ---
load_dotenv(override=True)                       # load .env; override=True = always use the newest values
connection_string = os.getenv("CON_STRING")      # MySQL address incl. password (from .env)
rapidapi_key = os.getenv("RAPIDAPI_KEY")         # AeroDataBox key (from .env)

# --- request headers ---
# RapidAPI expects the key in the HEADERS (an "envelope" around the request),
# not in the URL like OpenWeather did with appid=...
HEADERS = {
    "x-rapidapi-key": rapidapi_key,                   # who we are (our key)
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com"   # which API on RapidAPI we want to talk to
}

# --- cities from the database (source of truth) ---
cities_df = pd.read_sql("cities", con=connection_string)   # read the cities table into a DataFrame
cities_df                                                   # show it, to check that the connection works

,city_id,city_name,country,latitude,longitude
0,1,Berlin,Germany,52.52000,13.40500
1,2,Hamburg,Germany,53.55000,10.00000
2,3,Munich,Germany,48.13750,11.57500
3,4,Frankfurt,Germany,50.11056,8.68222


## 2. Airports
### 2.1 Explore: one test call for Berlin

In [4]:
# endpoint = the "address" of the specific service we want: search airports by location
url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

# parameters = the details of our question to the API
params = {
    "lat": 52.52,                 # latitude of Berlin (later we'll take this from cities_df)
    "lon": 13.405,                # longitude of Berlin
    "radiusKm": 50,               # only airports within 50 km of the city centre
    "limit": 10,                  # at most 10 airports in the answer
    "withFlightInfoOnly": "true"  # only airports that have flight schedules (skip tiny airfields)
}

# send the request: URL + headers (our key) + parameters (our question)
response = requests.get(url, headers=HEADERS, params=params)

print(response.status_code)       # 200 = success, 401/403 = problem with the key, 429 = too many requests
response.json()                   # convert the answer from JSON text into a Python dictionary and show it

200


{'searchBy': {'lat': 52.52, 'lon': 13.405},
 'count': 1,
 'items': [{'icao': 'EDDB',
   'iata': 'BER',
   'name': 'Berlin Brandenburg',
   'shortName': 'Brandenburg',
   'municipalityName': 'Berlin',
   'location': {'lat': 52.35139, 'lon': 13.493889},
   'countryCode': 'DE',
   'timeZone': 'Europe/Berlin'}]}

### 2.2 Airports pipeline

In [ ]:
def get_airports(cities_df):
    """Find airports (with flight data) near each city in cities_df.
    Returns one row per airport, linked to its city by city_id."""

    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"   # endpoint: search airports by location
    rows = []                                                            # one dictionary per airport

    for _, city in cities_df.iterrows():                 # go through the cities one row (= one city) at a time
        params = {
            "lat": city["latitude"],                     # city centre latitude, from the database
            "lon": city["longitude"],                    # city centre longitude, from the database
            "radiusKm": 50,                              # search radius around the centre
            "limit": 10,                                 # at most 10 airports per city
            "withFlightInfoOnly": "true"                 # only airports that have flight schedules
        }
        response = requests.get(url, headers=HEADERS, params=params)   # one API call per city

        if response.status_code != 200:                  # if something went wrong for this city...
            print(f"⚠️ Skipped {city['city_name']}: status {response.status_code}")
            continue                                     # ...skip it and go to the next city

        for airport in response.json().get("items", []): # "items" = list of airports; [] if the key is missing
            rows.append({
                "airport_icao":      airport["icao"],                  # e.g. 'EDDB' (always present)
                "airport_iata":      airport.get("iata"),              # e.g. 'BER'; None if missing
                "airport_name":      airport["name"],                  # e.g. 'Berlin Brandenburg'
                "airport_latitude":  airport["location"]["lat"],       # nested: location -> lat
                "airport_longitude": airport["location"]["lon"],       # nested: location -> lon
                "time_zone":         airport.get("timeZone"),          # e.g. 'Europe/Berlin'
                "city_id":           city["city_id"]                   # link to our cities table (added by us!)
            })

    if not rows:                                         # no airport found for any city in this run
        print("ℹ️ No airports found.")
        return pd.DataFrame()                            # return an empty table instead of crashing

    airports_df = pd.DataFrame(rows)                     # list of dictionaries -> table
    # the same airport can be found for two nearby cities in one run -> keep only the first
    # (airport_icao is our primary key and must be unique)
    airports_df = airports_df.drop_duplicates(subset="airport_icao")
    return airports_df


def load_new_airports(airports_df):
    """Send only airports that are not in the airports table yet (protects the ICAO primary key)."""

    airports_from_sql = pd.read_sql("airports", con=connection_string)    # what is already stored in MySQL

    # ~ = NOT  ->  keep the rows whose ICAO code is NOT already in the database
    new_airports_df = airports_df[~airports_df["airport_icao"].isin(airports_from_sql["airport_icao"])]

    if len(new_airports_df) > 0:                                          # only if there is something new
        new_airports_df.to_sql("airports",                 # MySQL table name (not the Python variable!)
                               if_exists="append",       # keep our table design, add rows
                               con=connection_string,  # the database address from .env
                               index=False)             # don't send pandas' row numbers 0,1,2,3

    print(f"✅ {len(new_airports_df)} new airports added")              # short report


def run_airports_pipeline():
    """Find airports only for cities that don't have one yet, and store them."""

    cities_df = pd.read_sql("cities", con=connection_string)            # all cities (fresh read)
    airports_from_sql = pd.read_sql("airports", con=connection_string)  # airports already stored

    # keep only cities whose city_id does NOT appear in the airports table yet
    cities_without_airport = cities_df[~cities_df["city_id"].isin(airports_from_sql["city_id"])]

    print(f"🏙️ {len(cities_df)} cities in the database, "
          f"{len(cities_without_airport)} without airport")

    if cities_without_airport.empty:                     # nothing to do -> no API calls at all
        print("✅ All cities already have airports. No API calls needed.")
        return                                           # stop here

    airports_df = get_airports(cities_without_airport)   # API calls ONLY for the missing cities
    if airports_df.empty:                                # the API found nothing for them
        return                                           # stop, nothing to load

    load_new_airports(airports_df)                       # store only new airports

### 2.3 Run (writes to MySQL)

In [11]:
# Remove the # to run: finds airports only for cities that don't have one yet (no API calls otherwise).
# run_airports_pipeline()

🏙️ 4 cities in the database, 0 without airport
✅ All cities already have airports. No API calls needed.


## 3. Flights
### 3.1 Explore: tomorrow's date and one test call (Berlin, morning)
The API allows at most 12 hours per request, so a full day needs two calls per airport.

In [12]:
from datetime import datetime, timedelta     # tools for dates and time differences

today = datetime.now()                       # current date and time on your computer
tomorrow = today + timedelta(days=1)         # timedelta = a duration; +1 day = tomorrow
tomorrow_str = tomorrow.strftime("%Y-%m-%d") # format as text the API understands, e.g. '2026-10-23'

print(tomorrow_str)

2026-10-23


In [13]:
icao = "EDDB"                                                   # Berlin Brandenburg (later: from the airports table)

# build the URL: airport + time window (local time of the airport)
url = (f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/"
       f"{tomorrow_str}T00:00/{tomorrow_str}T11:59")

params = {
    "direction": "Arrival",        # only arrivals (not departures)
    "withLeg": "true",             # include info about where the flight comes from
    "withCancelled": "false",      # skip cancelled flights
    "withCodeshared": "false",     # skip codeshare duplicates (one plane sold under several flight numbers)
    "withCargo": "false",          # skip cargo flights (no tourists on board!)
    "withPrivate": "false"         # skip private jets
}

response = requests.get(url, headers=HEADERS, params=params)   # ONE API call (run once!)
print(response.status_code)                                     # 200 = success
flights_json = response.json()                                  # JSON -> Python dictionary
flights_json.keys()                                             # which "drawers" does the answer have?

200


dict_keys(['arrivals'])

In [ ]:
arrivals = flights_json["arrivals"]    # the list of arriving flights in this 12-hour window
print(len(arrivals))                   # how many?
arrivals[0]                            # one flight: 'departure', 'arrival', 'number', 'airline', 'aircraft' ...

### 3.2 Flights pipeline

In [3]:
# ============================================================
# FLIGHTS PIPELINE
# 1. get_flights()          -> asks the API for TOMORROW's arrivals at each airport
# 2. load_flights()         -> appends them to the flights table
# 3. run_flights_pipeline() -> reads airports from MySQL and runs everything
# ============================================================


def get_flights(airports_df):
    """Get tomorrow's arriving flights for every airport in airports_df.
    One row per landing plane (codeshares excluded)."""

    rows = []                                             # empty list: one dictionary per flight

    # the API allows max 12 hours per request -> two windows cover a full day
    time_windows = [("00:00", "11:59"),                   # morning call
                    ("12:00", "23:59")]                   # afternoon/evening call

    for _, airport in airports_df.iterrows():             # one airport (row of the airports table) at a time
        icao = airport["airport_icao"]                    # e.g. 'EDDB'

        # "tomorrow" in the AIRPORT's own time zone (we stored it in the airports table!)
        # -> correct even for airports in other time zones (London, New York, ...)
        tomorrow = pd.Timestamp.now(tz=airport["time_zone"]).date() + pd.Timedelta(days=1)

        for start, end in time_windows:                   # two API calls per airport
            url = (f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/"
                   f"{tomorrow}T{start}/{tomorrow}T{end}")   # airport + time window in the URL

            params = {
                "direction": "Arrival",       # only arrivals: Gans cares about people COMING to the city
                "withLeg": "true",            # include the departure airport (where tourists come from)
                "withCancelled": "false",     # cancelled flights bring nobody
                "withCodeshared": "false",    # one plane sold by 2 airlines would otherwise appear twice
                "withCargo": "false",         # cargo planes carry no tourists
                "withPrivate": "false"        # private jets: few passengers, not Gans' target group
            }

            response = requests.get(url, headers=HEADERS, params=params)   # one API call

            if response.status_code != 200:               # safety: skip a failed window instead of crashing
                print(f"⚠️ Skipped {icao} {start}-{end}: status {response.status_code}")
                continue

            for item in response.json().get("arrivals", []):   # each item = one arriving flight
                rows.append({
                    "arrival_airport_icao":   icao,             # added by us: link to the airports table
                    # .get(..., {}) at each level = no crash if a piece of information is missing
                    "departure_airport_icao": item.get("departure", {}).get("airport", {}).get("icao"),
                    "scheduled_arrival_time": item.get("arrival", {}).get("scheduledTime", {}).get("local"),
                    "flight_number":          item.get("number"),
                    "airline":                item.get("airline", {}).get("name"),
                    "aircraft_model":         item.get("aircraft", {}).get("model"),
                    "arrival_terminal":       item.get("arrival", {}).get("terminal")
                })

    if not rows:                                          # nothing found at all -> empty table, no crash
        print("ℹ️ No flights found.")
        return pd.DataFrame()

    flights_df = pd.DataFrame(rows)                       # list of dictionaries -> table

    # a flight without arrival time is useless for Gans (and the column is NOT NULL in MySQL)
    flights_df = flights_df.dropna(subset=["scheduled_arrival_time"])

    # the API sends e.g. '2026-10-23 08:35+02:00'. The first 16 characters are always
    # 'YYYY-MM-DD HH:MM', so [:16] cuts off the time-zone offset safely,
    # also for negative offsets like '-05:00' (more robust than .str[:-6] or split("+"))
    flights_df["scheduled_arrival_time"] = pd.to_datetime(flights_df["scheduled_arrival_time"].str[:16])

    flights_df["retrieved_at"] = pd.Timestamp.now().floor("s")   # when we collected it (to the second)
    return flights_df


def load_flights(flights_df):
    """Append the flights to the flights table."""
    flights_df.to_sql("flights",                 # MySQL table name
                      if_exists="append",        # keep our table design, add rows
                      con=connection_string,     # database address from .env
                      index=False)               # don't send pandas' row numbers
    print(f"✅ {len(flights_df)} flights added")


def run_flights_pipeline():
    """Read our airports from MySQL, get tomorrow's arrivals, store them."""
    airports_df = pd.read_sql("airports", con=connection_string)    # source of truth: our airports
    print(f"🛫 {len(airports_df)} airports in the database "
          f"-> {len(airports_df) * 2} API calls")                     # 2 calls per airport (2 x 12 h)
    flights_df = get_flights(airports_df)
    if flights_df.empty:                                             # nothing to store -> stop
        return
    load_flights(flights_df)

### 3.3 Dry run (does NOT write to MySQL)

In [32]:
# dry run: get tomorrow's arrivals without writing to MySQL (8 API calls -> run once!)
airports_df = pd.read_sql("airports", con=connection_string)    # our airports from the database
flights_df = get_flights(airports_df)
flights_df["arrival_airport_icao"].value_counts()               # arrivals per airport

arrival_airport_icao
EDDF    628
EDDM    482
EDDB    301
EDDH    191
Name: count, dtype: int64

In [33]:
flights_df.head(10)       # do the times, airlines and models look plausible?

,arrival_airport_icao,departure_airport_icao,scheduled_arrival_time,flight_number,airline,aircraft_model,arrival_terminal,retrieved_at
0,EDDB,LTBJ,2026-10-23 06:05:00,XQ 966,Sun Express,Boeing 737 MAX 8,1,2026-10-22 15:31:48
1,EDDB,LTAJ,2026-10-23 06:45:00,XQ 1766,Sun Express,Boeing 737 MAX 8,1,2026-10-22 15:31:48
2,EDDB,ZBAA,2026-10-23 06:45:00,HU 489,Hainan,A330-900 neo,1,2026-10-22 15:31:48
3,EDDB,LROP,2026-10-23 07:05:00,W4 3109,Aero Services Executive,Airbus A321-200,2,2026-10-22 15:31:48
4,EDDB,OMAA,2026-10-23 07:15:00,DE 2553,Condor,Airbus A320 NEO,1,2026-10-22 15:31:48
5,EDDB,LZIB,2026-10-23 07:15:00,W6 7037,Wizz Air,Airbus A321 NEO,2,2026-10-22 15:31:48
6,EDDB,KEWR,2026-10-23 07:15:00,UA 962,United,Boeing 767-400ER,1,2026-10-22 15:31:48
7,EDDB,OTHH,2026-10-23 07:25:00,QR 79,Qatar,Boeing 787-9,1,2026-10-22 15:31:48
8,EDDB,LFSB,2026-10-23 07:30:00,U2 1187,easyJet,Airbus A320 NEO,1,2026-10-22 15:31:48
9,EDDB,LQTZ,2026-10-23 07:45:00,W6 4275,Wizz Air,Airbus A321 NEO,2,2026-10-22 15:31:48


In [34]:
flights_df.info()         # are scheduled_arrival_time and retrieved_at "datetime64"? any missing values?

<class 'pandas.DataFrame'>
RangeIndex: 1602 entries, 0 to 1601
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   arrival_airport_icao    1602 non-null   str           
 1   departure_airport_icao  1601 non-null   str           
 2   scheduled_arrival_time  1602 non-null   datetime64[us]
 3   flight_number           1602 non-null   str           
 4   airline                 1602 non-null   str           
 5   aircraft_model          1598 non-null   str           
 6   arrival_terminal        1597 non-null   str           
 7   retrieved_at            1602 non-null   datetime64[us]
dtypes: datetime64[us](2), str(6)
memory usage: 100.3 KB


### 3.4 Run (writes to MySQL)

In [4]:
# Remove the # to run: 8 API calls (4 airports x 2 time windows) + appends tomorrow's arrivals.
# run_flights_pipeline()

🛫 4 airports in the database -> 8 API calls
✅ 1491 flights added
